# Kansas City Pipe & Supply — Managed Runtime
### Notebook 2 of the Deploy & Monitor arc: *let Google run the box*

In the Cloud Run demo **we** built the box: a FastAPI wrapper, a Dockerfile, scaling flags, a secret, an auth policy. This notebook deploys the **same agent file** to Google's managed agent runtime, where we hand over a Python object and Google builds and runs the box.

> **Naming note (fall 2026):** Google renamed *Vertex AI* to **Gemini Enterprise Agent Platform**, and *Vertex AI Agent Engine* to **Agent Runtime**. The Python SDK and most tutorials still say `vertexai` / `agent_engines`. Same product, two names; expect both in docs and in student questions.

| | Cloud Run (what we did) | Agent Runtime (this notebook) |
|---|---|---|
| You provide | Container (Dockerfile + web server) | A Python object with a `query()` method |
| HTTP API | You write it (`main.py`) | Generated for you |
| Scaling knobs | `--concurrency`, `--min/--max-instances` | `container_concurrency`, `min/max_instances` |
| Secrets | `--set-secrets` → env var | `env_vars={"X": {"secret": ..., "version": ...}}` → env var |
| Agent extras | Build your own | Sessions, Memory Bank, Evaluation (built in) |
| Best for | Any app; full control | Agents in Google's ecosystem; less ops |

**Analogy:** Cloud Run is racking your own router in a colo. Agent Runtime is a managed SD-WAN service: the provider runs the box, you configure policy. The physics (queue depth, number of paths, upstream rate limits) didn't change. Only who operates the box did.

> ⏱️ **Deploying takes about 5–10 minutes.** For class, run through Step 5 **before** the session, save the resource name it prints, and reconnect in Step 6.

## Prerequisites (one time, in Cloud Shell or here)

1. A GCP project with billing enabled. The **same project** as the Cloud Run demo is ideal.
2. The `anthropic-api-key` secret in Secret Manager. If you ran `DEPLOY.md` step 1, it already exists and we reuse it.
3. Colab Secret `ANTHROPIC_API_KEY` for the local test in Step 3.

> ⚠️ **If the install cell prints import errors or version warnings:** *Runtime ▸ Restart session*, then run the cells again from the top (skip the install cell).

In [ ]:
%pip install -q "google-cloud-aiplatform[agent_engines]==2.7.0" langgraph==1.2.18 langchain-anthropic==1.7.6 cloudpickle pydantic

## Step 1 · Configure and authenticate

In [ ]:
import os

PROJECT_ID     = "your-project-id"                 # ← change
LOCATION       = "us-central1"
STAGING_BUCKET = f"gs://{PROJECT_ID}-agent-staging"  # created automatically if it doesn't exist
MODEL          = "claude-haiku-4-5-20251001"
SECRET_NAME    = "anthropic-api-key"              # same secret the Cloud Run service uses

USE_LANGSMITH  = False   # set True after you create the 'langsmith-api-key' secret (DEPLOY.md step 6)

def _load(name, required=True):
    """Colab Secrets first, then environment variables."""
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = os.getenv(name)
    if value:
        os.environ[name] = value
    elif required:
        raise RuntimeError(f"Missing secret: {name}. Add it in Colab Secrets (🔑) and enable notebook access.")
    return bool(value)

_load("ANTHROPIC_API_KEY")          # for the LOCAL test only, never shipped
os.environ["ANTHROPIC_MODEL"] = MODEL

try:
    from google.colab import auth
    auth.authenticate_user()        # pop-up sign-in to your Google account
    print("Authenticated with Google Cloud")
except ImportError:
    print("Not in Colab: using `gcloud auth application-default login` credentials")

In [ ]:
# Enable the APIs, then let the platform's service agent read the Anthropic secret.
# The deployed agent runs as the "Reasoning Engine Service Agent", not as you.
!gcloud config set project {PROJECT_ID} --quiet
!gcloud services enable aiplatform.googleapis.com storage.googleapis.com secretmanager.googleapis.com --quiet

PROJECT_NUMBER = !gcloud projects describe {PROJECT_ID} --format="value(projectNumber)"
PROJECT_NUMBER = PROJECT_NUMBER[0]
SERVICE_AGENT  = f"service-{PROJECT_NUMBER}@gcp-sa-aiplatform-re.iam.gserviceaccount.com"
print("Service agent:", SERVICE_AGENT)

for secret in [SECRET_NAME] + (["langsmith-api-key"] if USE_LANGSMITH else []):
    !gcloud secrets add-iam-policy-binding {secret} --member="serviceAccount:{SERVICE_AGENT}" --role="roles/secretmanager.secretAccessor" --quiet --format=none
print("Secret access granted")

> If the binding fails with *"service account does not exist"*, the service agent hasn't been created yet. It appears the first time the API is used. Run Step 5 once (it will fail on secret access), then re-run this cell and Step 5 again.

## Step 2 · Same agent file as Cloud Run

Byte for byte the same `plumbing_agent.py` as Notebook 1 and the `deploy/` folder. We ship it to Agent Runtime as an **extra package**, the way the Dockerfile `COPY`'d it into the container.

In [ ]:
%%writefile plumbing_agent.py
"""
plumbing_agent.py — Kansas City Pipe & Supply: counter-desk quoting agent.

ONE file, used three ways:
  1. Imported in the Colab notebook  (build + try it)
  2. Imported by main.py in Cloud Run (deploy + scale)
  3. Traced/evaluated by LangSmith    (monitor + evaluate)

Graph:  START -> extract -> (lookup) -> respond -> END
  extract  : LLM reads the customer message, picks SKUs + quantities (structured output)
  lookup   : plain Python — stock, price, delivery. No LLM. Deterministic = testable.
  respond  : LLM writes the reply using ONLY the lookup facts.

Env vars:
  ANTHROPIC_API_KEY   required
  ANTHROPIC_MODEL     optional, default claude-haiku-4-5-20251001
  ANTHROPIC_MAX_RETRIES optional, default 2 (set 0 to see raw 429s in the load test)
  ANTHROPIC_TEMPERATURE optional, unset by default (some newer models reject it)
  LANGSMITH_TRACING / LANGSMITH_API_KEY / LANGSMITH_PROJECT  optional (Block 2)
"""

from __future__ import annotations

import os
import time
from typing import Literal, Optional, TypedDict

from langgraph.graph import END, START, StateGraph
from langsmith import traceable
from pydantic import BaseModel, Field

DEFAULT_MODEL = "claude-haiku-4-5-20251001"

# ---------------------------------------------------------------------------
# 1. The "database" — a dict. In production this is an inventory API or SQL.
# ---------------------------------------------------------------------------
CATALOG: dict[str, dict] = {
    "PEX-12-ELB":   {"name": '1/2" PEX crimp elbow, 90°, brass',        "price": 1.89,  "stock": 600},
    "PEX-34-ELB":   {"name": '3/4" PEX crimp elbow, 90°, brass',        "price": 2.79,  "stock": 25},
    "PEX-12-TEE":   {"name": '1/2" PEX crimp tee, brass',               "price": 2.29,  "stock": 400},
    "PEX-34-TEE":   {"name": '3/4" PEX crimp tee, brass',               "price": 3.49,  "stock": 150},
    "PEX-12-100R":  {"name": '1/2" PEX-A pipe, 100 ft roll, red (hot)', "price": 54.00, "stock": 40},
    "PEX-12-100B":  {"name": '1/2" PEX-A pipe, 100 ft roll, blue (cold)', "price": 54.00, "stock": 38},
    "PEX-34-100R":  {"name": '3/4" PEX-A pipe, 100 ft roll, red (hot)', "price": 89.00, "stock": 12},
    "RING-12-100":  {"name": '1/2" copper crimp rings, bag of 100',     "price": 18.50, "stock": 80},
    "TOOL-CRIMP":   {"name": 'PEX crimp tool, 1/2" and 3/4" jaws',      "price": 64.99, "stock": 9},
    "HEAT-CABLE-6": {"name": "Self-regulating pipe heat cable, 6 ft",   "price": 39.95, "stock": 0},
    "FOAM-12-6":    {"name": '1/2" foam pipe insulation, 6 ft',         "price": 2.49,  "stock": 1200},
    "PTC-34-CPL":   {"name": '3/4" push-to-connect coupling',           "price": 9.98,  "stock": 60},
    "VALVE-34":     {"name": '3/4" brass ball valve, full port',        "price": 14.75, "stock": 45},
}

BULK_TIERS = [(100, 0.10), (50, 0.05)]  # (min qty, discount) — checked top-down


# ---------------------------------------------------------------------------
# 2. Deterministic tools. @traceable makes each one its own span in LangSmith.
# ---------------------------------------------------------------------------
@traceable(name="check_stock")
def check_stock(sku: str, qty: int) -> dict:
    on_hand = CATALOG[sku]["stock"]
    ship_now = min(on_hand, qty)
    return {"on_hand": on_hand, "ship_now": ship_now, "backorder": qty - ship_now}


@traceable(name="quote_price")
def quote_price(sku: str, qty: int) -> dict:
    unit = CATALOG[sku]["price"]
    discount = next((d for min_q, d in BULK_TIERS if qty >= min_q), 0.0)
    total = round(unit * qty * (1 - discount), 2)
    return {"unit_price": unit, "discount": discount, "line_total": total}


@traceable(name="delivery_estimate")
def delivery_estimate(ship_now: int, backorder: int) -> str:
    if backorder == 0:
        return "next business day"
    if ship_now == 0:
        return "backordered — about 5 business days"
    return f"{ship_now} next business day, remaining {backorder} in about 5 business days"


# ---------------------------------------------------------------------------
# 3. Structured output the extract step must return.
# ---------------------------------------------------------------------------
class LineRequest(BaseModel):
    sku: Optional[str] = Field(description="Exact SKU from the catalog, or null if we don't carry it")
    description: str = Field(description="What the customer asked for, in their words")
    quantity: int = Field(ge=1, description="Quantity requested; 1 if not stated")


class Extraction(BaseModel):
    intent: Literal["order_inquiry", "product_question", "out_of_scope"]
    items: list[LineRequest] = Field(default_factory=list)


class AgentState(TypedDict, total=False):
    message: str
    intent: str
    items: list[dict]
    lines: list[dict]
    order_total: float
    reply: str
    input_tokens: int
    output_tokens: int


# ---------------------------------------------------------------------------
# 4. LLM plumbing (lazy, so importing this file never needs an API key).
# ---------------------------------------------------------------------------
_llm = None


def get_llm():
    global _llm
    if _llm is None:
        from langchain_anthropic import ChatAnthropic

        kwargs = dict(
            model=os.getenv("ANTHROPIC_MODEL", DEFAULT_MODEL),
            max_tokens=800,
            max_retries=int(os.getenv("ANTHROPIC_MAX_RETRIES", "2")),
        )
        # Some newer Claude models reject `temperature`, so it's opt-in only.
        if os.getenv("ANTHROPIC_TEMPERATURE"):
            kwargs["temperature"] = float(os.environ["ANTHROPIC_TEMPERATURE"])
        _llm = ChatAnthropic(**kwargs)
    return _llm


def _catalog_text() -> str:
    return "\n".join(f"{sku}: {v['name']}" for sku, v in CATALOG.items())


EXTRACT_PROMPT = """You are the order desk at Kansas City Pipe & Supply.
Read the customer message and identify each product they want and how many.
Map each product to an exact SKU from this catalog. If we don't carry it, set sku to null.

CATALOG:
{catalog}

Intent rules:
- order_inquiry: they want price, availability, or to buy specific items
- product_question: they ask whether we carry something or which item fits
- out_of_scope: anything unrelated to buying plumbing supplies"""

REPLY_PROMPT = """You are the order desk at Kansas City Pipe & Supply. Write a short, friendly reply
to the customer. Use ONLY the facts in ORDER LINES — never invent prices, stock, or dates.
Mention bulk discounts when applied. For items we don't carry, say so plainly.
Plain text, no markdown tables, under 120 words.

CUSTOMER MESSAGE:
{message}

ORDER LINES:
{lines}

ORDER TOTAL: ${total:.2f}"""

OUT_OF_SCOPE_REPLY = (
    "Thanks for reaching out! I can help with plumbing supplies — pricing, stock, "
    "and delivery. What can I look up for you?"
)


def _tokens(msg) -> tuple[int, int]:
    u = getattr(msg, "usage_metadata", None) or {}
    return u.get("input_tokens", 0), u.get("output_tokens", 0)


# ---------------------------------------------------------------------------
# 5. Graph nodes.
# ---------------------------------------------------------------------------
def extract(state: AgentState) -> AgentState:
    llm = get_llm().with_structured_output(Extraction, include_raw=True)
    result = llm.invoke([
        ("system", EXTRACT_PROMPT.format(catalog=_catalog_text())),
        ("human", state["message"]),
    ])
    parsed: Extraction = result["parsed"]
    tin, tout = _tokens(result["raw"])
    return {
        "intent": parsed.intent,
        "items": [i.model_dump() for i in parsed.items],
        "input_tokens": state.get("input_tokens", 0) + tin,
        "output_tokens": state.get("output_tokens", 0) + tout,
    }


def lookup(state: AgentState) -> AgentState:
    lines = []
    for item in state.get("items", []):
        sku, qty = item.get("sku"), item["quantity"]
        if sku not in CATALOG:
            lines.append({"sku": None, "requested": item["description"], "quantity": qty,
                          "status": "not carried"})
            continue
        stock = check_stock(sku, qty)
        price = quote_price(sku, qty)
        lines.append({
            "sku": sku, "name": CATALOG[sku]["name"], "quantity": qty, **price, **stock,
            "delivery": delivery_estimate(stock["ship_now"], stock["backorder"]),
        })
    total = round(sum(l.get("line_total", 0) for l in lines), 2)
    return {"lines": lines, "order_total": total}


def respond(state: AgentState) -> AgentState:
    if state.get("intent") == "out_of_scope":
        return {"reply": OUT_OF_SCOPE_REPLY}
    msg = get_llm().invoke(REPLY_PROMPT.format(
        message=state["message"],
        lines="\n".join(str(l) for l in state.get("lines", [])) or "(none)",
        total=state.get("order_total", 0.0),
    ))
    tin, tout = _tokens(msg)
    return {
        "reply": msg.content if isinstance(msg.content, str) else str(msg.content),
        "input_tokens": state.get("input_tokens", 0) + tin,
        "output_tokens": state.get("output_tokens", 0) + tout,
    }


def route_after_extract(state: AgentState) -> str:
    return "respond" if state.get("intent") == "out_of_scope" else "lookup"


def build_graph():
    g = StateGraph(AgentState)
    g.add_node("extract", extract)
    g.add_node("lookup", lookup)
    g.add_node("respond", respond)
    g.add_edge(START, "extract")
    g.add_conditional_edges("extract", route_after_extract, {"lookup": "lookup", "respond": "respond"})
    g.add_edge("lookup", "respond")
    g.add_edge("respond", END)
    return g.compile()


graph = build_graph()


def run_agent(message: str) -> dict:
    """Convenience wrapper: one customer message in, one result dict out (with timing)."""
    t0 = time.perf_counter()
    out = graph.invoke({"message": message})
    out["latency_ms"] = round((time.perf_counter() - t0) * 1000)
    out["model"] = os.getenv("ANTHROPIC_MODEL", DEFAULT_MODEL)
    return out


## Step 3 · The adapter: a class with `query()`

This replaces `main.py`. Agent Runtime's contract is simple: give it an object with a `query` method, and it becomes an API endpoint.

| Method | When it runs | Our use |
|---|---|---|
| `__init__` | On your laptop/Colab, before upload | Store config only (the model name) |
| `set_up()` | Once, inside each cloud container at startup | Import the agent, build the graph |
| `query()` | On every request | Run one customer message |

**Why `set_up` matters:** the object is *pickled* (serialized) and uploaded. API clients and network connections don't survive pickling, so anything heavy gets created in `set_up`, on the cloud side. It's the equivalent of a router loading its config at boot rather than shipping with live sessions.

In [ ]:
class PlumbingAgentApp:
    """Kansas City Pipe & Supply quote agent, packaged for Agent Runtime."""

    def __init__(self, model: str = "claude-haiku-4-5-20251001"):
        self.model = model          # plain data only: this object gets pickled
        self._agent = None

    def set_up(self):
        import os
        os.environ["ANTHROPIC_MODEL"] = self.model
        import plumbing_agent       # shipped via extra_packages
        self._agent = plumbing_agent

    def query(self, message: str) -> dict:
        """Answer one customer message: SKUs, prices, stock, delivery, and a reply."""
        if self._agent is None:
            self.set_up()
        out = self._agent.run_agent(message)
        keep = ["reply", "intent", "lines", "order_total",
                "input_tokens", "output_tokens", "latency_ms", "model"]
        return {k: out.get(k) for k in keep}   # must be JSON-serializable

### Test locally first, always

Same habit as running `uvicorn` locally before `gcloud run deploy`. If it fails here, it'll fail in the cloud, just 10 minutes slower.

In [ ]:
local_app = PlumbingAgentApp(model=MODEL)
local_app.set_up()

r = local_app.query("Need 40 of the 3/4 PEX elbows and 120 half-inch tees. When can I get them?")
print(r["reply"])
print(f"\nTotal ${r['order_total']:.2f} · {r['input_tokens']}+{r['output_tokens']} tokens · {r['latency_ms']} ms")

## Step 4 · The deployment config: same knobs, different dashboard

In [ ]:
import cloudpickle, pydantic

requirements = [
    "google-cloud-aiplatform[agent_engines]==2.7.0",
    "langgraph==1.2.18",
    "langchain-anthropic==1.7.6",
    f"cloudpickle=={cloudpickle.__version__}",   # must match the version that pickled the object
    f"pydantic=={pydantic.VERSION}",
]

env_vars = {
    # A SecretRef, not a value. The key is fetched from Secret Manager at deploy time
    # and injected as an env var. It never sits in the notebook or the uploaded bundle.
    "ANTHROPIC_API_KEY": {"secret": SECRET_NAME, "version": "latest"},
    "ANTHROPIC_MODEL": MODEL,
}
if USE_LANGSMITH:
    env_vars |= {
        "LANGSMITH_TRACING": "true",
        "LANGSMITH_PROJECT": "plumbing-agent-runtime",
        "LANGSMITH_API_KEY": {"secret": "langsmith-api-key", "version": "latest"},
    }

config = {
    "display_name": "plumbing-agent",
    "description": "Kansas City Pipe & Supply quote agent (LangGraph + Claude)",
    "staging_bucket": STAGING_BUCKET,
    "requirements": requirements,
    "extra_packages": ["plumbing_agent.py"],
    "env_vars": env_vars,
    # ── Scaling: the same physics as Cloud Run ──────────────────────────────
    "min_instances": 1,                                   # warm instance = no cold start (costs money while idle)
    "max_instances": 3,                                   # Cloud Run: --max-instances
    "resource_limits": {"cpu": "2", "memory": "2Gi"},
    "container_concurrency": 5,                           # Cloud Run: --concurrency. Google suggests 2 × CPU + 1
}

for k, v in config.items():
    if k != "env_vars":
        print(f"{k:22} {v}")
print(f"{'env_vars':22} {list(env_vars)}  (secrets shown by name only)")

## Step 5 · Deploy (≈ 5–10 min)

Behind this one call, Google:
1. Pickles `PlumbingAgentApp(...)` and uploads it, `plumbing_agent.py`, and the requirements to the staging bucket
2. **Builds a container image for you** (the Dockerfile you didn't write)
3. Starts instances, runs `set_up()` in each, and wires `query()` to an authenticated HTTPS endpoint

> Deploy a **fresh** instance, not `local_app`. `local_app` already ran `set_up()`, so it holds a live API client that shouldn't be pickled.

In [ ]:
import time
import vertexai

client = vertexai.Client(project=PROJECT_ID, location=LOCATION)

t0 = time.time()
remote = client.agent_engines.create(
    agent=PlumbingAgentApp(model=MODEL),
    config=config,
)
RESOURCE_NAME = remote.api_resource.name
print(f"Deployed in {(time.time() - t0) / 60:.1f} min")
print("RESOURCE_NAME =", RESOURCE_NAME)
print("👆 Save this. Step 6 reconnects with it in a later session.")

## Step 6 · Reconnect (start here in class)

In [ ]:
import vertexai
client = vertexai.Client(project=PROJECT_ID, location=LOCATION)

# RESOURCE_NAME = "projects/123456789/locations/us-central1/reasoningEngines/1234567890"   # ← paste if new session
remote = client.agent_engines.get(name=RESOURCE_NAME)
print("Connected:", remote.api_resource.display_name)

# Bonus for the room: list everything deployed in this project/region
for ae in client.agent_engines.list():
    print(" -", ae.api_resource.display_name, "→", ae.api_resource.name.split("/")[-1])

## Step 7 · Call it

Notice what we *didn't* write: no FastAPI, no route, no auth code. `remote.query(...)` is an authenticated HTTPS call generated from our `query()` method's signature.

In [ ]:
r = remote.query(message="Pipe burst. Need 40 of the 3/4 inch PEX elbows and a 100 ft roll of 3/4 red PEX. When can I get them?")
print(r["reply"])
print(f"\nTotal ${r['order_total']:.2f} · {r['input_tokens']}+{r['output_tokens']} tokens · server-side {r['latency_ms']} ms")

### A mini surge

Same idea as `loadtest.py`, smaller. Watch the *round-trip* time vs. the agent's own `latency_ms`: the difference is network plus platform overhead.

In [ ]:
import time, statistics
from concurrent.futures import ThreadPoolExecutor

messages = [
    "Price on 120 half-inch crimp tees?",
    "Do you have heat tape? Need 3 for a crawl space.",
    "Got any 1 inch copper sweat elbows?",
    "How much for 60 of the 3/4 push-to-connect couplings?",
    "Need a 3/4 ball valve and 10 of the 3/4 tees.",
    "Need 2 bags of 1/2 crimp rings and a crimp tool.",
] * 2   # 12 requests

def timed(m):
    t0 = time.perf_counter()
    try:
        out = remote.query(message=m)
        return ("ok", (time.perf_counter() - t0) * 1000, out["latency_ms"])
    except Exception as e:
        return (type(e).__name__, (time.perf_counter() - t0) * 1000, None)

t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=6) as pool:
    results = list(pool.map(timed, messages))
wall = time.perf_counter() - t0

ok = [r for r in results if r[0] == "ok"]
print(f"{len(ok)}/{len(results)} succeeded in {wall:.1f}s")
if ok:
    print(f"Round trip p50: {statistics.median(r[1] for r in ok)/1000:.1f}s   "
          f"Agent work p50: {statistics.median(r[2] for r in ok)/1000:.1f}s")
errors = [r[0] for r in results if r[0] != "ok"]
if errors:
    print("Errors:", errors)

## Discussion: which one would you pick?

| Question | Points to Cloud Run | Points to Agent Runtime |
|---|---|---|
| Is it a plain web app, or an agent? | Any HTTP app, any framework | Agents, especially ADK / LangGraph |
| Do you need Sessions or long-term memory? | Build it (Redis, Firestore…) | Built in |
| Does a non-Python client (website, n8n) call it? | Any HTTP client, plain JSON | Possible, but the SDK path is smoothest |
| Who's on call? | You own the container | Google owns more of the stack |
| Portability | A container runs anywhere | Tied to Google's platform |
| Model choice | Anything | Anything (we used Claude), smoothest with Gemini |

**For PMs, the real question:** what do you want your team spending time on, operating infrastructure or improving the agent? There's no wrong answer, only a trade.

## Step 8 · Clean up

`min_instances=1` means an instance stays warm, and **bills**, until you delete it.

In [ ]:
# remote.delete(force=True)   # force=True also deletes child resources (sessions, memories)
# print("Deleted", RESOURCE_NAME)